<a href="https://colab.research.google.com/github/nieblasIX/IX-SierraGorda-Fragmentacion_Conectividad/blob/main/IX_SierraGorda_Fragmentacion_Conectividad.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==============================================================================
# PROYECTO: Corredor Bioculutural del Jaguar - Sierra Gorda Hidalguense
# TAREA: Análisis de Fragmentación y Conectividad Funcional
# ==============================================================================

# --- 1. CONFIGURACIÓN ROBUSTA DEL ENTORNO ---
paquetes <- c("terra", "landscapemetrics", "dplyr", "ggplot2", "tidyr")
for (pkg in paquetes) {
  if (!require(pkg, character.only = TRUE)) {
    install.packages(pkg, dependencies = TRUE)
    library(pkg, character.only = TRUE)
  }
}

# --- 2. CARGA Y VERIFICACIÓN DE DATOS ---
ruta_raster <- "/content/01_RF_LaMision_30m.tif"

if (file.exists(ruta_raster)) {
  mapa_rf <- rast(ruta_raster)
  print("Raster cargado con éxito.")
} else {
  stop("ALERTA: El archivo no existe en /content. Súbelo antes de continuar.")
}

# --- 3. PROCESAMIENTO MÉTRICO (EPSG:32614) ---
print("Reproyectando a UTM Zona 14N para cálculos métricos...")
mapa_rf_metric <- project(mapa_rf, "EPSG:32614", method = "near")

# --- 4. CÁLCULO DE MÉTRICAS (Nivel Clase y Parche) ---
print("Calculando métricas de conectividad...")
# Clase (Resumen municipal)
met_clase <- calculate_lsm(mapa_rf_metric, classes_max = 10,
                           what = c("lsm_c_tca", "lsm_c_enn_mn", "lsm_c_cohesion"))
# - TCA: Total Core Area (¿Hay suficiente área profunda para cazar?)
# - ENN_MN: Euclidean Nearest-Neighbor (¿Qué tan lejos está el próximo refugio?)
# - COHESION: Índice de Cohesión (¿Está roto el corredor?)


# 5. Mostrar los resultados limpios en pantalla
resultados_limpios <- met_clase %>% select(metric, value)
print("--- MÉTRICAS DE CONECTIVIDAD: LA MISIÓN ---")
print(resultados_limpios)

#Para asegurarnos de que las métricas de conectividad sean precisas, recalcularemos las métricas después de proyectar el mapa a un Sistema de Referencia de Coordenadas (CRS) métrico adecuado.

# Parche (Datos para estadística y mapa)
met_parche <- calculate_lsm(mapa_rf_metric, classes_max = 10,
                            what = c("lsm_p_area", "lsm_p_enn"))

# Guardar CSV
write.csv(met_clase, "Metricas_Clase_LaMision.csv", row.names = FALSE)
write.csv(met_parche, "Metricas_Parche_LaMision.csv", row.names = FALSE)

# --- 5. ESPACIALIZACIÓN (Generación de .tif para QGIS) ---
print("Espacializando métricas para QGIS...")
mapas_spat <- spatialize_lsm(mapa_rf_metric, what = c("lsm_p_area", "lsm_p_enn"))

# Guardar los rásters de métricas
writeRaster(mapas_spat$lsm_p_area, "Mapa_Area_Parches_LaMision.tif", overwrite=TRUE)
writeRaster(mapas_spat$lsm_p_enn, "Mapa_Aislamiento_ENN_LaMision.tif", overwrite=TRUE)

# --- 6. ANÁLISIS ESTADÍSTICO (Correlación y Coherencia) ---
print("Generando análisis de correlación...")
datos_corr <- met_parche %>%
  pivot_wider(names_from = metric, values_from = value)

grafico_corr <- ggplot(datos_corr, aes(x = area, y = enn)) +
  geom_point(alpha = 0.5, aes(size = area, color = area)) +
  geom_smooth(method = "lm", color = "red", se = TRUE) +
  scale_color_viridis_c() +
  labs(title = "Coherencia del Paisaje: Tamaño vs Aislamiento",
       subtitle = "Municipio: La Misión | Clase: Bosque",
       x = "Área del Parche (ha)", y = "Distancia al Vecino (m)") +
  theme_minimal()

ggsave("Histograma_Correlacion_Jaguar.png", grafico_corr, width = 8, height = 6)
print(grafico_corr)

Loading required package: terra

Warning message in library(package, lib.loc = lib.loc, character.only = TRUE, logical.return = TRUE, :
“there is no package called ‘terra’”
Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘proxy’, ‘e1071’, ‘wk’, ‘lazyeval’, ‘sp’, ‘classInt’, ‘s2’, ‘units’, ‘crosstalk’, ‘leaflet.providers’, ‘png’, ‘raster’, ‘tinytest’, ‘ncdf4’, ‘sf’, ‘deldir’, ‘XML’, ‘leaflet’


Warning message in install.packages(pkg, dependencies = TRUE):
“installation of package ‘ncdf4’ had non-zero exit status”
terra 1.9.11

Loading required package: landscapemetrics

Warning message in library(package, lib.loc = lib.loc, character.only = TRUE, logical.return = TRUE, :
“there is no package called ‘landscapemetrics’”
Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘rex’, ‘abind’, ‘RcppArmadillo’, ‘covr’, ‘stars’


Loading required package: dplyr


Att

ERROR: Error: ALERTA: El archivo no existe en /content. Súbelo antes de continuar.


In [ ]:
install.packages('IRkernel')

Once installed, you can change the runtime type by going to `Runtime > Change runtime type` and selecting `R` as the runtime type.